# Trockenmasse-Indizes im SWIR 2 (CAI, NDTI)

**Autor:** jd · **Datum:** 2026-10-29 · **Issue:** –

**Hypothese:** Ein Index für trockenes Pflanzenmaterial trennt abgereifte Bestände vom Boden und senkt Winterweizen → Baumwolle (AEZ 5, Mature_Senesc/Emerge_VEarly) sowie Critical → Mature_Senesc. Das SWIR 2 ist für Mais und Baumwolle die wichtigste Region, hat aber keinen Katalog-Index. Der Standard-CAI braucht das leere Band X2002, daher eine verschobene Variante (2063/2123/2184 nm) plus NDTI (1650/2215 nm).

**Ablauf:** Jede Variante bekommt dasselbe `C` × `gamma`-Raster wie die Referenz (`combined_svm_aez_month`). Entschieden wird nur per CV; validiert wird nur, wenn der Gewinn `MIN_CV_GAIN` (0.005) erreicht.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from awp2.config import CROP_COL, MIN_CV_GAIN, STAGE_COL
from awp2.data import load_split
from awp2.evaluation import compare_predictions, recall_by_class
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models.combined import make_combined_svm
from awp2.plots import plot_cv_scores, plot_metric_comparison, plot_recall_comparison
from awp2.preprocessing import PreprocessingConfig

# A · Läufe

In [ ]:
# reference: combined_svm_aez_month (d93cea89); every variant gets the same grid
REFERENCE = PreprocessingConfig(use_spectral_standard_scale=True, use_aez_month=True)
GRID = {"estimator__C": [100.0, 300.0, 1000.0], "estimator__gamma": [0.001, 0.003, 0.01]}
y_train = load_split().y_train
SLUG = "combined_svm_dry_matter"
TOPIC = "Trockenmasse-Indizes"
VARIANTS = {
    "reference": ("Referenz", REFERENCE),
    "dry_matter": ("+ CAI, NDTI", REFERENCE.model_copy(update={"use_dry_matter_indices": True})),
}

In [ ]:
searches = {
    label: tune(
        make_combined_svm(),
        TuneConfig(
            name=f"{SLUG}_{key}_search",
            approach="combined",
            description=f"{TOPIC}: {label}; RBF-SVM, gleiches Raster wie die Referenz.",
            param_grid=GRID,
            preprocessing=preprocessing,
        ),
    )
    for key, (label, preprocessing) in VARIANTS.items()
}
pd.DataFrame(
    {label: {"CV": result.best_score, **result.best_params} for label, result in searches.items()}
).T

# B · Erkenntnisse

## 1. CV-Score je Variante

In [ ]:
reference_cv = searches["Referenz"].best_score
ax = plot_cv_scores(
    {label: result.candidates for label, result in searches.items()}, reference=reference_cv
)
ax.set_title(f"{TOPIC}: bester CV-Score je Variante (gestrichelt: Referenz)")

## 2. Gepaart auf dem Trainingsteil (out-of-fold)

Gleiche Zeilen, beste Variante gegen die Referenz: Wie viele Zeilen trifft nur eine?

In [ ]:
variants = [label for label in searches if label != "Referenz"]
best_label = max(variants, key=lambda label: searches[label].best_score)
best = searches[best_label]
oof = {
    label: out_of_fold_predictions(searches[label].best_model, searches[label].config)
    for label in ("Referenz", best_label)
}
print(f"Beste Variante: {best_label}")
compare_predictions(y_train, oof["Referenz"], oof[best_label])

## 3. Zielfehler: Weizen → Baumwolle, Critical → Mature_Senesc

In [ ]:
def target_errors(pred: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "Weizen → Baumwolle": (
                (y_train[CROP_COL] == "winter_wheat") & (pred[CROP_COL] == "cotton")
            ).sum(),
            "Critical → Mature_Senesc": (
                (y_train[STAGE_COL] == "Critical") & (pred[STAGE_COL] == "Mature_Senesc")
            ).sum(),
        }
    )


pd.DataFrame({label: target_errors(pred) for label, pred in oof.items()})

## 4. Recall je Klasse: wo ändert sich etwas?

In [ ]:
plot_recall_comparison(
    {label: recall_by_class(y_train, pred) for label, pred in oof.items()}, top=10
)

## 5. Entscheidung

In [ ]:
gain = best.best_score - reference_cv
print(f"CV-Gewinn {gain:+.4f} (Schwelle {MIN_CV_GAIN})")
if gain >= MIN_CV_GAIN:
    validated = run(
        best.best_model,
        RunConfig(
            name=f"{SLUG}",
            approach="combined",
            description=f"{TOPIC}: {best_label}, einmal auf der Validierung.",
            preprocessing=best.config.preprocessing,
            tuning_run=best.run_id,
        ),
    )
    reference = run(
        searches["Referenz"].best_model,
        RunConfig(name="reference", approach="combined", preprocessing=REFERENCE, track=False),
    )
    display(plot_metric_comparison({"Referenz": reference.metrics, best_label: validated.metrics}))
else:
    print("Kein Gewinn über der Rauschschwelle – keine Validierung, Referenz bleibt.")

# Fazit

- CV 0.861 mit CAI (2063/2123/2184 nm) und NDTI gegen 0.864 der Referenz (−0.003) – **nicht gestützt**.
- Die Zielfehler bleiben: Weizen → Baumwolle 60 → 61, Critical → Mature_Senesc 38 → 40
  (out-of-fold). Trockenes Pflanzenmaterial im SWIR 2 trennt die hellen Weizenfelder nicht von
  Baumwolle; die SVM nutzt die SWIR-2-Bänder offenbar schon direkt.
- Keine Validierung; die Referenz bleibt. Schließt den offenen CAI-Punkt der Feature Importance.